In [1]:
using CSV, DataFrames, Random, Statistics
using GLM, DecisionTree, MLJ, MLJLinearModels


In [5]:
using CSV, DataFrames

# ---------------------------------------------------------
# 1. Cargar datos (penn_jae.dat)
# ---------------------------------------------------------
data = DataFrame(CSV.File(
    "C:/Users/VICTOR/Documents/GitHub/UP_courses/penn_jae.dat.txt",
    delim=' ',
    ignorerepeated=true
))

println("Columnas disponibles:")
println(names(data))

# Definir variables
y = log.(data.inuidur1)                # outcome
d = (data.tg .== 4) .* 1.0             # dummy de tratamiento (numérico)
X = select(data, Not([:inuidur1, :tg]))  # controles

# Convertir a matrices con dimensión (n,1)
y = reshape(y, :, 1)
d = reshape(d, :, 1)
X = Matrix(X)

println("Observaciones: $(size(X,1))")
println("Covariables: $(size(X,2))")


Columnas disponibles:
["abdt", "tg", "inuidur1", "inuidur2", "female", "black", "hispanic", "othrace", "dep", "q1", "q2", "q3", "q4", "q5", "q6", "recall", "agelt35", "agegt54", "durable", "nondurable", "lusd", "husd", "muld"]
Observaciones: 13913
Covariables: 21


In [6]:
# ---------------------------------------------------------
# 2. Función principal DML con cross-fitting
# ---------------------------------------------------------
function DML2_for_PLM(X, D, Y, dreg_fun, yreg_fun; nfold=10)
    n = size(X, 1)
    foldid = rand(1:nfold, n)
    ytil = zeros(n)
    dtil = zeros(n)

    for b in 1:nfold
        idx_train = findall(foldid .!= b)
        idx_test  = findall(foldid .== b)

        dfit = dreg_fun(X[idx_train, :], D[idx_train])
        yfit = yreg_fun(X[idx_train, :], Y[idx_train])

        dhat = predict(dfit, X[idx_test, :])
        yhat = predict(yfit, X[idx_test, :])

        dtil[idx_test] .= D[idx_test] .- dhat
        ytil[idx_test] .= Y[idx_test] .- yhat
    end

    df = DataFrame(resy = ytil, resD = dtil)
    reg = lm(@formula(resy ~ resD), df)
    coef_est = coef(reg)[2]
    se = stderror(reg)[2]

    println("Coef = $(round(coef_est, digits=4)), SE = $(round(se, digits=4))")
    return (; coef_est, se, ytil, dtil)
end

# ---------------------------------------------------------
# 3. Definir modelos base
# ---------------------------------------------------------

# OLS usando MLJLinearModels
ols_model() = LinearRegressor()

# Lasso (L1 penalización)
lasso_model() = LassoRegressor(lambda=0.01)

# Random Forest (DecisionTreeRegressor)
rf_model() = DecisionTreeRegressor(max_depth=5, n_subfeatures=0, min_samples_leaf=5)

# Helpers para entrenamiento/predicción
function fit_predict(model, X, y)
    mach = machine(model, X, y)
    fit!(mach)
    return mach
end

function predict(mach, X)
    ŷ = MLJ.predict(mach, X)
    return MLJBase.unwrap(ŷ)
end



predict (generic function with 1 method)

In [7]:

# ---------------------------------------------------------
# 4. Correr DML con varios learners
# ---------------------------------------------------------
println("\n--- DML con OLS ---")
res_OLS = DML2_for_PLM(X, d, y,
    (X,d)->fit_predict(ols_model(), X, d),
    (X,y)->fit_predict(ols_model(), X, y)
)

println("\n--- DML con Lasso ---")
res_Lasso = DML2_for_PLM(X, d, y,
    (X,d)->fit_predict(lasso_model(), X, d),
    (X,y)->fit_predict(lasso_model(), X, y)
)

println("\n--- DML con Random Forest ---")
res_RF = DML2_for_PLM(X, d, y,
    (X,d)->fit_predict(rf_model(), X, d),
    (X,y)->fit_predict(rf_model(), X, y)
)

println("\n--- DML con Mix (Lasso para D, RF para Y) ---")
res_Mix = DML2_for_PLM(X, d, y,
    (X,d)->fit_predict(lasso_model(), X, d),
    (X,y)->fit_predict(rf_model(), X, y)
)

# ---------------------------------------------------------
# 5. Resumen en tabla
# ---------------------------------------------------------
RMSE_D = [sqrt(mean(res_OLS.dtil.^2)),
          sqrt(mean(res_Lasso.dtil.^2)),
          sqrt(mean(res_RF.dtil.^2)),
          sqrt(mean(res_Mix.dtil.^2))]

RMSE_Y = [sqrt(mean(res_OLS.ytil.^2)),
          sqrt(mean(res_Lasso.ytil.^2)),
          sqrt(mean(res_RF.ytil.^2)),
          sqrt(mean(res_Mix.ytil.^2))]

models = ["OLS", "Lasso", "RF", "Mix"]
estimates = [res_OLS.coef_est, res_Lasso.coef_est, res_RF.coef_est, res_Mix.coef_est]
ses = [res_OLS.se, res_Lasso.se, res_RF.se, res_Mix.se]

results = DataFrame(Model=models, Estimate=estimates, SE=ses, RMSE_Y=RMSE_Y, RMSE_D=RMSE_D)
println("\n=== Resultados DML (Julia) ===")
println(results)


--- DML con OLS ---


┌ Warning: The number and/or types of data arguments do not match what the specified model
│ supports. Suppress this type check by specifying `scitype_check_level=0`.
│ 
│ Run `@doc MLJLinearModels.LinearRegressor` to learn more about your model's requirements.
│ 
│ Commonly, but non exclusively, supervised models are constructed using the syntax
│ `machine(model, X, y)` or `machine(model, X, y, w)` while most other models are
│ constructed with `machine(model, X)`.  Here `X` are features, `y` a target, and `w`
│ sample or class weights.
│ 
│ In general, data in `machine(model, data...)` is expected to satisfy
│ 
│     scitype(data) <: MLJ.fit_data_scitype(model)
│ 
│ In the present case:
│ 
│ scitype(data) = Tuple{AbstractMatrix{Count}, AbstractVector{Continuous}}
│ 
│ fit_data_scitype(model) = Tuple{Table{<:AbstractVector{<:Continuous}}, AbstractVector{Continuous}}
└ @ MLJBase C:\Users\VICTOR\.julia\packages\MLJBase\GY2fM\src\machines.jl:237


LoadError: UndefVarError: `fit!` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name may be made accessible by importing StatsBase in the current active module Main
Hint: a global variable of this name also exists in GLM.
Hint: a global variable of this name may be made accessible by importing ScikitLearnBase in the current active module Main
Hint: a global variable of this name also exists in DecisionTree.
Hint: a global variable of this name may be made accessible by importing MLJBase in the current active module Main
Hint: a global variable of this name also exists in MLJ.